# 📊 Gold Spot
Notebook 02: Modeling

## Preliminaries


In [1]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, TimeSeriesSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

from src.config import clean_data_file
from src.modeling_functions import *

In [2]:
# Open data file
df = pd.read_parquet(clean_data_file)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 874 entries, 0 to 873
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   date                         874 non-null    datetime64[ns]
 1   gold_return                  874 non-null    float64       
 2   oil_return                   874 non-null    float64       
 3   spy_return                   874 non-null    float64       
 4   real_yield_10y_lag1_diff1    874 non-null    float64       
 5   dxy_broad_goods_lag1_return  874 non-null    float64       
 6   target_gold_t1               873 non-null    float64       
 7   target_gold_binary_t1        873 non-null    float64       
 8   vix_index_lag1_return_Yeo    874 non-null    float64       
 9   month_of_year                874 non-null    int32         
dtypes: datetime64[ns](1), float64(8), int32(1)
memory usage: 65.0 KB


In [3]:
# Ensure order by date
df.sort_values(by='date', inplace=True)

# Split training and live data
target = 'target_gold_t1'
binary_target = "target_gold_binary_t1"


features_to_remove = [target]
X_train, X_live, y_train_raw, _ = split_last(df, label=binary_target, cols_excluded=features_to_remove)

# Cast y_train to int
y_train = y_train_raw.astype(int)

# Class balance in train
class_balance = y_train.value_counts(normalize=True).to_dict()
print(f"Class distribution in train: {class_balance}")

# Column transformer
cyclical_features = ["month_of_year"]
numeric_features = [col for col in X_train.select_dtypes(include=np.number) if col not in cyclical_features and col != 'date']

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cyclical", FunctionTransformer(encode_cyclical_month), cyclical_features),
    ]
)

# Base model pipeline
model_name = 'LogisticRegression'
pipe = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                C=0.1,               # Inverso de regularizacion: menor C = mas regularizacion
                l1_ratio=0,
                solver="lbfgs",
                class_weight="balanced", # Compensa leves sesgos alcistas/bajistas
                random_state=42,
            ),
        ),
    ]
)

# Training
pipe.fit(X_train, y_train)

# Live Predict
y_pred_class = pipe.predict(X_live)[0]
y_pred_prob = pipe.predict_proba(X_live)[0]  # [P(Down), P(Up)]

print("Training completed.")
print(f"Live Prediction: {'UP (1)' if y_pred_class == 1 else 'DOWN (0)'}")
print(f"Probabilities: P(Down) = {y_pred_prob[0]:.2%}, P(Up) = {y_pred_prob[1]:.2%}")

Class distribution in train: {1: 0.5601374570446735, 0: 0.43986254295532645}
Training completed.
Live Prediction: UP (1)
Probabilities: P(Down) = 48.14%, P(Up) = 51.86%


In [4]:
# Cross validation test
tscv = TimeSeriesSplit(n_splits=5)

cv_roc_auc = cross_val_score(
    estimator=pipe,
    X=X_train,
    y=y_train,
    cv=tscv,
    scoring="roc_auc",
    n_jobs=-1,
)

# Evaluacion con Accuracy Direccional
cv_accuracy = cross_val_score(
    estimator=pipe,
    X=X_train,
    y=y_train,
    cv=tscv,
    scoring="accuracy",
    n_jobs=-1,
)

print(f"\nROC-AUC CV:  {cv_roc_auc.mean():.4f} ± {cv_roc_auc.std():.4f}")
print(f"Accuracy CV: {cv_accuracy.mean():.2%} ± {cv_accuracy.std():.2%}")


ROC-AUC CV:  0.4893 ± 0.0555
Accuracy CV: 47.59% ± 6.31%
